# Interval evaluation for linear PyTorch networks

This notebook demonstrates the prototype `intervalNets` toolbox on several linear-network cases.

In [ ]:
import numpy as np
import torch
from torch import nn

from intervalnets import IntervalTensor, enable_interval_eval

torch.set_printoptions(precision=17)
enable_interval_eval()

## 1. Random linear network

A random affine network shows the general propagation path.

In [ ]:
torch.manual_seed(7)
random_model = nn.Sequential(
    nn.Linear(3, 4),
    nn.Linear(4, 2),
)
random_interval = IntervalTensor.from_bounds([0.0, -1.0, 2.0], [0.2, -0.8, 2.3])
random_output = random_model.eval(random_interval)
random_output

## 2. Zero network

Even a degenerate zero input produces a nonzero-width output interval because we expand point intervals outward to capture roundoff.

In [ ]:
zero_model = nn.Linear(3, 2)
with torch.no_grad():
    zero_model.weight.zero_()
    zero_model.bias.zero_()

zero_interval = IntervalTensor.point([0.0, 0.0, 0.0])
zero_output = zero_model.eval(zero_interval)
zero_output

## 3. Hand-computable linear network

This example is easy to verify analytically because it is purely affine.

In [ ]:
linear_model = nn.Linear(2, 1)
with torch.no_grad():
    linear_model.weight.copy_(torch.tensor([[2.0, -3.0]]))
    linear_model.bias.copy_(torch.tensor([0.5]))

linear_interval = IntervalTensor.from_bounds([1.0, 2.0], [1.5, 2.5])
linear_output = linear_model.eval(linear_interval)
linear_output

## 4. Identity-style sanity check

The network should preserve the input interval up to outward rounding when it acts like the identity map.

In [ ]:
identity_model = nn.Linear(2, 2)
with torch.no_grad():
    identity_model.weight.copy_(torch.eye(2))
    identity_model.bias.zero_()

identity_interval = IntervalTensor.from_bounds([-1.0, 4.0], [1.0, 5.0])
identity_output = identity_model.eval(identity_interval)
identity_output

## 5. Unsupported activation

Right now nonlinear activations are intentionally left unimplemented so their interval extensions can be added carefully later.

In [ ]:
unsupported_model = nn.Sequential(nn.Linear(2, 2), nn.ReLU())
try:
    unsupported_model.eval(IntervalTensor.point([0.0, 1.0]))
except NotImplementedError as exc:
    print(type(exc).__name__, exc)